# Scraping Instagram Google Play Store Reviews & Ratings

Project untuk scraping data rating dan review aplikasi Instagram dari Google Play Store.
Data dikumpulkan sebagai dataset untuk analisis teks pada mata kuliah **Pemrosesan Teks** (Semester 5).

---

## 1. Prerequisites & Installation

Library yang dibutuhkan:
- `google-play-scraper` — scraping review dari Google Play tanpa API key
- `pandas` — data processing dan export CSV

In [ ]:
!pip install google-play-scraper pandas

## 2. Import Libraries & Configuration

In [ ]:
import time
import re
import pandas as pd
from google_play_scraper import app, reviews, Sort

# ── Configuration ─────────────────────────────────────────────
APP_ID = 'com.instagram.android'
TARGET_COUNT = 15000
LANG = 'en'
COUNTRY = 'us'
BATCH_SIZE = 200
SLEEP_BETWEEN_BATCHES = 0.5
SLEEP_ON_ERROR = 2

DESIRED_COLUMNS = [
    'reviewId', 'userName', 'score',
    'thumbsUpCount', 'reviewCreatedVersion', 'at', 'content'
]

print("Configuration loaded.")
print(f"Target: {TARGET_COUNT} reviews for {APP_ID}")

## 3. Fetch App Metadata

Mengambil informasi umum aplikasi Instagram dari Google Play Store.

In [ ]:
app_info = app(APP_ID, lang=LANG, country=COUNTRY)

print(f"App Title       : {app_info.get('title')}")
print(f"Average Rating  : {app_info.get('score', 0):.2f} / 5.0")
print(f"Total Ratings   : {app_info.get('ratings')}")
print(f"Total Reviews   : {app_info.get('reviews')}")

## 4. Scrape 15,000 Reviews

Menggunakan paginated `reviews()` dengan batch size 200 untuk menghindari rate limit.

In [ ]:
all_reviews = []
token = None
start = time.time()

print(f"[*] Scraping {TARGET_COUNT} reviews...")

while len(all_reviews) < TARGET_COUNT:
    remaining = TARGET_COUNT - len(all_reviews)
    count = min(BATCH_SIZE, remaining)

    try:
        batch, token = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=count,
            continuation_token=token
        )

        if not batch:
            print("[*] No more reviews available.")
            break

        all_reviews.extend(batch)
        if len(all_reviews) % 1000 == 0 or len(all_reviews) >= TARGET_COUNT:
            print(f"    Progress: {len(all_reviews)} / {TARGET_COUNT}")

        if not token:
            print("[*] Reached end of available reviews.")
            break

        time.sleep(SLEEP_BETWEEN_BATCHES)

    except Exception as e:
        print(f"[!] Batch error: {e}")
        time.sleep(SLEEP_ON_ERROR)
        continue

elapsed = time.time() - start
print(f"\n[+] Scraped {len(all_reviews)} reviews in {elapsed:.1f}s")

## 5. Export Raw Data (15,000 Reviews)

Menyimpan seluruh data mentah ke CSV.

In [ ]:
df_raw = pd.DataFrame(all_reviews)

# Select & reorder columns
cols = [c for c in DESIRED_COLUMNS if c in df_raw.columns]
df_raw = df_raw[cols]

# Export
RAW_OUTPUT = 'instagram_reviews_raw_15000.csv'
df_raw.to_csv(RAW_OUTPUT, index=False, sep=';', encoding='utf-8-sig')

print(f"[+] Exported '{RAW_OUTPUT}' ({len(df_raw)} rows, {len(df_raw.columns)} cols)")
print(f"\nScore distribution:")
print(df_raw['score'].value_counts().sort_index())
print(f"\nNull counts:")
print(df_raw.isnull().sum())
print(f"\nSample data:")
df_raw.head()

## 6. Data Cleaning

Proses cleaning untuk mendapatkan ~3000 review kritik yang bermakna:

1. **Filter score 1-3** — hanya review negatif/kritik
2. **Remove emoji-only** — review yang isinya hanya emoji tanpa teks substantif
3. **Remove terlalu pendek** — minimal 3 kata dan 12 karakter setelah strip emoji
4. **Remove spam** — karakter berulang 5+ kali, konten non-alfabet > 60%
5. **Normalize whitespace** — collapse multiple spaces

In [ ]:
# ── Cleaning functions ────────────────────────────────────────

MIN_WORDS = 3
MIN_CLEAN_CHARS = 12

EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\U00002702-\U000027B0"
    "\U0000FE00-\U0000FE0F"
    "\U0000200D"
    "\U000023E9-\U000023F3"
    "\U0000231A-\U0000231B"
    "\U00002328"
    "\U000023CF"
    "\U000023F8-\U000023FA"
    "\U00002934-\U00002935"
    "\U000025AA-\U000025AB"
    "\U000025B6"
    "\U000025C0"
    "\U000025FB-\U000025FE"
    "\U00002600-\U000027BF"
    "\U00002B05-\U00002B07"
    "\U00002B1B-\U00002B1C"
    "\U00002B50"
    "\U00002B55"
    "\U00003030"
    "\U0000303D"
    "\U00003297"
    "\U00003299"
    "]+", flags=re.UNICODE
)

SPAM_RE = re.compile(r'(.)\1{4,}')


def strip_emojis(text):
    return EMOJI_RE.sub('', text)


def is_meaningful(text):
    if pd.isna(text):
        return False
    t = str(text).strip()
    if not t:
        return False

    no_emoji = strip_emojis(t).strip()
    if len(no_emoji) < MIN_CLEAN_CHARS:
        return False

    words = no_emoji.split()
    if len(words) < MIN_WORDS:
        return False

    alpha_chars = sum(1 for c in no_emoji if c.isalpha())
    if alpha_chars < len(no_emoji) * 0.4:
        return False

    if SPAM_RE.search(no_emoji):
        cleaned = SPAM_RE.sub('', no_emoji).strip()
        if len(cleaned) < MIN_CLEAN_CHARS:
            return False

    return True


def clean_text(text):
    t = str(text).strip()
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

print("Cleaning functions defined.")

## 7. Apply Cleaning & Export Clean Data

In [ ]:
TARGET_CLEAN = 3000

# Deduplicate
df = df_raw.drop_duplicates(subset='reviewId').copy()
print(f"After dedup: {len(df)}")

# Filter critical reviews (score 1-3)
df_crit = df[df['score'] <= 3].copy()
print(f"Critical (score 1-3): {len(df_crit)}")

# Quality filter
mask = df_crit['content'].apply(is_meaningful)
df_clean = df_crit[mask].copy()
print(f"After quality filter: {len(df_clean)}")

# Clean text
df_clean['content'] = df_clean['content'].apply(clean_text)
df_clean = df_clean[df_clean['content'].str.len() > 0]
print(f"After text cleaning: {len(df_clean)}")

# Trim to target
if len(df_clean) >= TARGET_CLEAN:
    df_final = df_clean.head(TARGET_CLEAN).reset_index(drop=True)
    print(f"Trimmed to target: {TARGET_CLEAN}")
else:
    df_final = df_clean.reset_index(drop=True)
    print(f"[!] Only {len(df_final)} clean reviews (target was {TARGET_CLEAN})")

# Export
CLEAN_OUTPUT = 'instagram_reviews_cleaned_3000.csv'
df_final.to_csv(CLEAN_OUTPUT, index=False, sep=';', encoding='utf-8-sig')
print(f"\n[+] Exported '{CLEAN_OUTPUT}' ({len(df_final)} rows)")

## 8. Validation & Summary

In [ ]:
print("=" * 50)
print("VALIDATION REPORT")
print("=" * 50)
print(f"Rows    : {len(df_final)}")
print(f"Columns : {list(df_final.columns)}")
print(f"\nScore distribution:")
print(df_final['score'].value_counts().sort_index())
print(f"\nNull counts:")
print(df_final.isnull().sum())

empty_content = (df_final['content'].isna() | (df_final['content'] == '')).sum()
print(f"\nEmpty content rows: {empty_content}")

print(f"\n{'=' * 50}")
print("SAMPLE REVIEWS")
print("=" * 50)
for i, row in df_final.head(10).iterrows():
    print(f"[{row['score']}★] {str(row['content'])[:150]}")
    print()

## 9. Preview Clean Dataset

In [ ]:
df_final.head(20)

In [ ]:
df_final.describe()